## How To Start
In this notebook we will install the darts-devkit package and explore basic structure and navigation of the dataset in devkit.

### How to install
Since darts-devkit uses Python 3.14 the easiers way to install it is to use virtual environments. We recommend using uv project manager.
Below are commands that prepare python environment.
* curl -LsSf https://astral.sh/uv/install.sh | sh
* uv python install 3.14 --default
* uv init
#### Run Jupyter Notebook
To run jupyter notebook with just prepared virtual environment run:
* uv add pip jupyterlab ipykernel
* uv run python -m ipykernel install --user --name py314-venv --display-name "Python 3.14 (venv)"
* uv run jupyter notebook

Then with opened notebook choose just installed kernel with Python 3.14.
#### Install devkit
To install devkit run below command:

In [ ]:
pip install darts-devkit

### Schema
Below schema shows structure of present tables in dataset with connections between them. For clarity of the schema only fields that are used to traverse to different tables are present.

There are four main parts in dataset, each marked by different colour. 
* Blue part has data about annotations.
* Green part has data about scenes.
* Yellow part has data about sensors that were recording scenes.
* Orange part has data about INS readings and additional scene metadata.

![darts_devkit](../images/darts_devkit.jpg)

#### General traversal

This notebook assumes that test version of dataset is present in /data/darts directory.
First we will take a look at how to move between tables. Every table present in schema above has two methods.
* all method returns every row of this table
* get method returs row of this table with given token, token field is unique for every table

In [ ]:
from darts_devkit import DARTS
darts = DARTS("/data/darts", "test")
scenes = darts.scene.all()
scene = darts.scene.get(scenes[0].token)

As shown in schema above we can use tokens from one table to get to specific rows of data in different tables. For example we can get first and last sample from our scene.

In [ ]:
first_sample = darts.sample.get(scene.first_sample_token)
last_sample = darts.sample.get(scene.last_sample_token)

#### Scene data

##### scene
Every scene has name, description and number of samples present.

In [ ]:
print(scene)

##### sample
Every sample is a keyframe. This means that it has annotations and that sample_data rows that point to it all have nearly identical timestamps with this sample. All timestamps in all tables are in microseconds. Data field is a dictionary with keys being channel names. Those names are present in sensor table and consist of (CAM_FRONT_TELE, CAM_FRONT_LEFT, CAM_FRONT, CAM_FRONT_RIGHT, CAM_BACK_LEFT, CAM_BACK, CAM_BACK_RIGHT, LIDAR_TOP). Anns field is a tuple with 3D annotations present in LIDAR.

In [ ]:
print(first_sample)

##### sample_data
Sample data is a collection of data from one sensor, for example CAM_FRONT. Sample data consists of filename (path to file with data from this sensor), fileformat (png for cameras and pcd.bin for lidar), width and height (0 for lidar), is_key_frame booelan value, checksum of a file, channel and modality of this sensor. 

In [ ]:
first_sample_data_cam_front = darts.sample_data.get(first_sample.data.get("CAM_FRONT"))
print(first_sample_data_cam_front)

##### ego_pose
Ego pose is a pose of ego vehicle with respect to the global coordinate system. Translation is in meters in format (x,y,z), rotation is Quaternion in format (w, x, y, z).

In [ ]:
ego_pose = darts.ego_pose.get(first_sample_data_cam_front.ego_pose_token)
print(ego_pose)

#### Sensors data

##### calibrated_sensor
Calibrated sensor is a table with data about used sensor in dataset with respect to the ego pose coordinate system. Translation is meters in format (x, y, z), rotation is Quaternion in format (w, x ,y, z) and camera_intristic is a 3x3 Matrix for camera sensor and empty list for lidar sensor.

In [ ]:
calibrated_sensor = darts.calibrated_sensor.get(first_sample_data_cam_front.calibrated_sensor_token)
print(calibrated_sensor)

##### sensor
Dataset was created with seven cameras and four lidars merged into one. They have one row each in sensor table. Every element consists of channel (CAM_FRONT_TELE, CAM_FRONT_LEFT, CAM_FRONT, CAM_FRONT_RIGHT, CAM_BACK_LEFT, CAM_BACK, CAM_BACK_RIGHT, LIDAR_TOP) and modality (camera, lidar).

In [ ]:
sensor = darts.sensor.get(calibrated_sensor.sensor_token)
print(sensor)

#### Annotations data

##### sample_annotation
Sample annotation is 3D annotation on one frame/sample. Sample annotation uses global coordinate system. Translation is in meters in format (x, y, z), rotation is Quaternion in format (w, x ,y ,z) and size is in meters in format (x, y, z). Every sample annotation also has information about the number of lidar points in the cuboid and list of attributes. 

In [ ]:
sample_annotation = darts.sample_annotation.get(first_sample.anns[0])
print(sample_annotation)

##### instance
Instance is a collection of 3D sample annotations of the same object in a scene in lidar space. It consists of number of annotations.

In [ ]:
instance = darts.instance.get(sample_annotation.instance_token)
print(instance)

##### attribute
Attribute is a property of an instance. Such attributes can be change in different frames. For example car can have opened dors only in half of scene. Every attribute has a name and description. Name consists of name of attribute and what classes can have this attribute. For example multi_track_vehicle.open_trunk is an attribute of open_trunk for classes with prefix multi_track_vehicle.

In [ ]:
attribute = darts.attribute.get(sample_annotation.attribute_tokens[0])
print(attribute)

##### category
One object during its existance in a scene can have only one category that can not change. That is why in contrast to attributes it has relationship to instance. It has a name and description. Similarly to attribute its name consists of proper name and metacategories. For example multi_track_vehicle.car is car that belongs to multi_track_vehicle metacategory.

In [ ]:
category = darts.category.get(instance.category_token)
print(category)

##### sample_annotation_2d
Similarly to sample_annotation in lidar space every camera can have 2D annotations. Which camera recorded this sample can be infered from calibrated_sensor. If there is no 3D annotations for this object, instance record will not be present and using instace_token will end in exception. Corners field is in format (min_x, min_y, max_x, max_y).

In [ ]:
sample_annotation_2d = darts.sample_annotation_2d.get(first_sample_data_cam_front.anns[0])
print(sample_annotation_2d)

##### instance_2d
Instance 2d is a collection of 2D sample annotations of the same object in a scene in one of a camera space. Meaning one object can have up to 8 instances (one lidar instance and seven instance_2d for every camera). It consists of number of annotations.

In [ ]:
instance_2d = darts.instance_2d.get(sample_annotation_2d.instance_2d_token)
print(instance_2d)

#### INS readings and additional scene metadata data

##### ins
Ins has additional information about characteristics of car movement during scene recording. It consists of latitude, longitude, roll, pitch, yaw, velocity and acceleration in x, y, z, and angular velocity of car body in x, y, z.

In [ ]:
ins = darts.ins.get(first_sample.ins_token)
print(ins)

##### scene_metadata
Scene metadata consists of human readeble characteristics of recorded scene such as road geometry, traffic infrastructure, environment conditions or even vertical signs.

In [ ]:
scene_metadata = darts.scene_metadata.get(scene.scene_metadata_token)
print(scene_metadata)

#### Public methods

##### verify_integrity
You can verify the integrity of images and lidar point clouds with this simple method.

In [ ]:
darts.verify_integrity()

##### filter_scenes
With this method you can create new DARTS object with subset of scenes. It filters scenes based on their metadata. Below example returns DARTS object with scenes that do not have Y intersection or that have trucks or cyclist. A list is treated as a disjunction, not a conjunction. It means that at least one of the values of the list must be present in the scene metadata. In the example above, a track or a cyclist should be present in the recording, not both of them.

In [ ]:
query = {
    "or": [
        {"not": {"intersection_y": 1}},
        {"traffic_participants": ["trucks", "cyclist"]},
    ]
}
darts_filtered = darts.filter_scenes(query)

##### get_annotations_from_samples
With this method we can easily get lidar annotations of multiple samples.

In [ ]:
annotations = darts.get_annotations_from_samples([first_sample, last_sample])

##### get_annotations_2d_from_sample_datas
We can do the same with camera annotations and sample datas.

In [ ]:
annotations_2d = darts.get_annotations_2d_from_sample_datas([first_sample_data_cam_front])

##### get_lidar_pointcloud
With this method we can get LidarPointCloud object which points are in (x, y, z, intensity) format. Only works for sample data with lidar modality.

In [ ]:
lidar_sample_data = darts.sample_data.get(first_sample.data.get("LIDAR_TOP"))
point_cloud = darts.get_lidar_pointcloud(lidar_sample_data)
print(point_cloud.points.T)

##### get_image
This method allows as to get image in PIL.ImageFile.ImageFile format. Only works sample data with camera modality.

In [ ]:
image = darts.get_image(first_sample_data_cam_front)
print(image)

##### get_samples_from_scene
This helper method returns all samples from a scene in chronological order.

In [ ]:
samples = darts.get_samples_from_scene(scene.token)
print(samples[0])

##### get_sensor_from_annotation_2d
This method helps us with getting to know from which camera is this annotation.

In [ ]:
sensor = darts.get_sensor_from_annotation_2d(sample_annotation_2d.token)
print(sensor)

##### get_category_from_annotation
This method helps us with getting to know which category has this annotation.

In [ ]:
category = darts.get_category_from_annotation(sample_annotation.token)
print(category)

##### get_category_from_annotation_2d
We can check the same with camera annotations.

In [ ]:
category = darts.get_category_from_annotation_2d(sample_annotation_2d.token)
print(category)